#  OpenMP Programming
# 1. Introduction
* [OpenMP](http://openmp.org/) is the de fact programming model for multicore environment
* More recently, it supports GPU offloading
* We are going to learn OpenMP both for CPU (multicore) and GPU programming
* A very lawyerish / cryptic [OpenMP ver 5.1 specification](https://www.openmp.org/spec-html/5.1/openmp.html) I can't say you should read first but have to tell you about anyway (there is a spec doc for the newer version, 5.2, which I feel guilty even putting a link to here)

# 2. How to work on this notebook
* Read text, modify cells if necessary, and execute them
* Many cells can be executed without modifying them
* Cells that need your work are typically preceeded with <font color="green">text colored green explaining what you need to do</font>
* Problems with non-trivial work have a section header starting with <font color="green">"Problem: X"</font>

# 3. Jupyter AI
* You can ask Jupyter AI when you need a help
* Open `pd01_omp.chat` in this folder
* Choose `OpenCode` in Persona Chooser
* You can try several models, but `LiteLLM (taulec)/GPT-5.3 Codex (UTokyo Azure)` is recommended for now
* It is instructed to help you but not to give you answer directly to exercise problems; do not try to circumvent it

# 4. Setting up compilers
* We use [NVIDIA HPC SDK 25.9](https://docs.nvidia.com/hpc-sdk/index.html) (`nvc` and `nvc++`) and [LLVM](https://llvm.org/) (`clang` and `clang++`) as our C/C++ compilers. Both support OpenMP GPU offloading, although their feature support differs.

## 4-1. NVIDIA
- Load the `nvidia/25.9` module to use the NVIDIA compilers (`nvc` and `nvc++`).
- To start from a clean module environment, run:
```
module purge
module load nvidia/25.9
```

- Check that the following commands:

In [ ]:
%%bash
module purge
module load nvidia/25.9
which nvc
which nvc++

produce paths like these:
```
/work/opt/local/aarch64/cores/nvidia/25.9/Linux_aarch64/25.9/compilers/bin/nvc
/work/opt/local/aarch64/cores/nvidia/25.9/Linux_aarch64/25.9/compilers/bin/nvc++
```

- Save the following program:

In [ ]:
%%writefile hello_nvidia.cc
#include <stdio.h>

int main() {
  printf("hello world\n");
  return 0;
}

- Compile it:

In [ ]:
%%bash
module purge
module load nvidia/25.9
nvc++ hello_nvidia.cc -o hello_nvc

- Run it. Make sure you execute `%load_ext miyabi` cell before executing `%%sub` cell; You may add `--no` after `%%sub` to run on the login node instead of submitting it to a compute node.

In [ ]:
%load_ext miyabi

In [ ]:
%%sub
./hello_nvc

- Note: only short, lightweight programs should be run on the login node.

## 4-2. LLVM
- Load the `cuda/12.9` module and add `/work/gt81/share/env/llvm/bin` to `PATH` to use the LLVM compilers (`clang` and `clang++`).
- To start from a clean module environment, run:
```
module purge
module load cuda/12.9
PATH=/work/gt81/share/env/llvm/bin:$PATH
```

- Check that the following commands:

In [ ]:
%%bash
module purge
module load cuda/12.9
PATH=/work/gt81/share/env/llvm/bin:$PATH
which clang
which clang++

produce these paths:
```
/work/gt81/share/env/llvm/bin/clang
/work/gt81/share/env/llvm/bin/clang++
```

- Save the following program:

In [ ]:
%%writefile hello_llvm.cc
#include <stdio.h>

int main() {
  printf("hello world\n");
  return 0;
}

- Compile it:

In [ ]:
%%bash
module purge
module load cuda/12.9
PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ hello_llvm.cc -o hello_clang

- Run it. Add `--no` after `%%sub` to run on the login node.

In [ ]:
%%sub
./hello_clang

## 4-3. A setting to choose one
- For the examples below, run this setup at the start of a `%%bash` cell. It makes both compiler toolchains available:


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH

- We show commands for both compilers because their OpenMP feature support differs; some examples may work with only one of them.
- The examples below use LLVM by default. You can choose NVIDIA instead and should be prepared to switch compilers if an example is unsupported.

- If you prefer to work in a terminal, run the relevant module and `PATH` commands in each new shell, or add them to a Miyabi shell startup file such as `~/.bashrc`.


# 5. Compiling and running OpenMP programs
## 5-1. Overview
* For `clang`/`clang++`, use the `-fopenmp` option.
* For `nvc`/`nvc++`, use the `-mp` option.
* Set `OMP_NUM_THREADS` when running the executable to choose the requested team size.


In [ ]:
%%writefile omp_hello_demo.cc
#include <stdio.h>

int main() {
  printf("hello\n");
#pragma omp parallel
  printf("world\n");
  printf("good bye\n");
  return 0;
}

## 5-2. Compiling

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -fopenmp omp_hello_demo.cc -o omp_hello_demo
# nvc++ -mp omp_hello_demo.cc -o omp_hello_demo

### 5-2-1. Useful options
* `clang`/`clang++`
  * `-Wall` enables many useful warnings.
  * `-O3` enables aggressive optimization.
* `nvc`/`nvc++`
  * `-Wall` enables many useful warnings.
  * `-O4` enables aggressive optimization.

## 5-3. Running
- Set environment variable `OMP_NUM_THREADS` to the number of threads created by `#pragma omp parallel`
- <font color="green">Exercise: Change the number of threads and observe the difference.</font>


In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_hello_demo

# 6. Parallel construct `#pragma omp parallel`
* Together with the `for` construct introduced next, [#pragma omp parallel](https://www.openmp.org/spec-html/5.2/openmpse57.html) is one of the basic OpenMP constructs.
* It creates a team of threads. `OMP_NUM_THREADS` usually controls the requested team size; each thread executes the associated structured block.
  - See [Determining the Number of Threads for a parallel Region](https://www.openmp.org/spec-html/5.2/openmpsu95.html) for details.
* Only the structured block immediately following the directive is executed by the team of threads.
* The associated statement can be a compound statement or a function call, so each thread can execute multiple statements.

* <font color="green">Exercise: Modify the following program so that both "world" and "good bye" are printed once by each thread.</font>


In [ ]:
%%writefile omp_hello.cc
#include <stdio.h>

int main() {
  printf("hello\n");
#pragma omp parallel
  printf("world\n");
  printf("good bye\n");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_hello.cc -o omp_hello
# nvc++ -O4 -Wall -mp omp_hello.cc -o omp_hello

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_hello

# 7. `omp_get_num_threads()` and `omp_get_thread_num()`
* When threads execute a construct with `#pragma omp parallel`, they are in a _parallel region_ and form a _team_.

* While a thread is executing a parallel region,
  * [omp_get_num_threads()](https://www.openmp.org/spec-html/5.2/openmpsu138.html) returns the number of threads in the team.
  * [omp_get_thread_num()](https://www.openmp.org/spec-html/5.2/openmpsu140.html) returns the calling thread's ID in the team (`0` through `omp_get_num_threads() - 1`).
* Include `<omp.h>` to use these functions and other OpenMP API routines.

* <font color="green">Exercise: Modify the following program so that each thread prints its ID and the number of threads in the team, as shown below.</font>

```
hello
0/5 world
4/5 world
1/5 world
3/5 world
2/5 world
good bye
```

* The order of the lines may differ.
* In principle, output from different threads may even interleave within a line.


In [ ]:
%%writefile omp_hello_id.cc
#include <stdio.h>

int main() {
  printf("hello\n");
#pragma omp parallel
  printf("world\n");
  printf("good bye\n");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_hello_id.cc -o omp_hello_id
# nvc++ -O4 -Wall -mp omp_hello_id.cc -o omp_hello_id

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_hello_id

# 8. The `for` construct: `#pragma omp for`
## 8-1. Basics
* `#pragma omp parallel` creates a team of threads that execute the same structured block.
* By itself, `#pragma omp parallel` does not divide loop iterations among threads.
* To use multiple cores effectively, divide work among threads with a work-sharing construct.
* [`#pragma omp for`](https://www.openmp.org/spec-html/5.2/openmpsu105.html) distributes loop iterations among threads in the current team.

* <font color="green">Exercise: Run the program and observe which thread executes each iteration. Then change the iteration count and thread count, and predict how the iterations will be mapped.</font>


In [ ]:
%%writefile omp_for.cc
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
#pragma omp parallel
  {
    printf("I am thread %d in a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
#pragma omp for
    for (int i = 0; i < 24; i++) {
      usleep(100 * 1000 * i);
      printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
      fflush(stdout);
    }
  }
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_for.cc -o omp_for
# nvc++ -O4 -Wall -mp omp_for.cc -o omp_for

In [ ]:
%%sub
OMP_NUM_THREADS=4 ./omp_for

## 8-2. Restrictions of `#pragma omp for`
* There are several restrictions on the kinds of loops to which `#pragma omp for` can be applied.
* First and foremost, iterations may execute in any order and concurrently, so loop correctness must not depend on sequential execution.
* For example, you cannot apply `#pragma omp for` to the following loop
```
for (long i = 2; i < n; i++) {
  a[i] = a[i - 2] + a[i - 1];
}
```
because this loop inherently relies on serial execution of iterations (i = 0, 1, 2, ...).
* For example, executing the iteration `i=10`, which reads `a[8]` and `a[9]`, before the iteration `i=9`, would result in a result different from sequential execution.
* In a nutshell, each iteration must be able to compute _independently_ (without assuming execution status of other iterations at all).

* Besides, the loop must also satisfy the [canonical loop form](https://www.openmp.org/spec-html/5.2/openmpsu28.html).
* In short, it should look like `for (var = _init_; var < _limit_; var += _inc_)`, where `_init_`, `_limit_`, and `_inc_` are loop-invariant (they do not change during the loop).

## 8-3. Combined parallel + for pragma
* `#pragma omp parallel` and `#pragma omp for` are often used together.
* When `#pragma omp for` immediately follows `#pragma omp parallel`, the constructs can be combined as `#pragma omp parallel for`.


In [ ]:
%%writefile omp_parallel_for.cc
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel for
  for (int i = 0; i < 24; i++) {
    usleep(100 * 1000 * i);     /* sleep 100 x i milliseconds */
    printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
    fflush(stdout);
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_parallel_for.cc -o omp_parallel_for
# nvc++ -O4 -Wall -mp omp_parallel_for.cc -o omp_parallel_for

In [ ]:
%%sub
OMP_NUM_THREADS=4 ./omp_parallel_for

## 8-4. Collapse clause
* `#pragma omp for` can use a [collapse clause](https://www.openmp.org/spec-html/5.2/openmpsu30.html) to distribute iterations from a supported nested loop.
* With `#pragma omp for collapse(2)`, OpenMP distributes the iterations of the following doubly nested loop among the threads. The loop nest must be perfectly nested and rectangular.
* A perfectly nested loop is a loop nest in which each outer loop contains only the next loop. For example:
```
for (i = 0; i < 100; i++) {
  for (j = 0; j < 100; j++) {
    S(i,j);
  }
}
```
is perfectly nested, whereas
```
for (i = 0; i < 100; i++) {
  S;
  for (j = 0; j < 100; j++) {
    T;
  }
}
```
is not.
* Conceptually, a perfectly nested loop can be flattened by a mechanical transformation:
```
for (ij = 0; ij < 100 * 100; ij++) {
  i = ij / 100;
  j = ij % 100;
  S(i,j);
}
```
* A rectangular loop is one whose inner-loop iteration counts do not depend on outer-loop indices. For example,
```
for (i = 0; i < 100; i++) {
  for (j = 0; j < i; j++) {
    S(i,j);
  }
}
```
is not a rectangular loop.
* The `collapse` clause distributes iterations across nested loops. Use tasks for more general forms of nested parallelism.

# 9. Scheduling a work-sharing for loop
* The scheduling policy determines how loop iterations are partitioned across threads. You can select a policy with the [schedule clause](https://www.openmp.org/spec-html/5.2/openmpsu107.html).

## 9-1. Recording iteration -> thread mapping
* The following program records which thread executes each loop iteration and when.
* For each sampled value of $a$, it counts the iterations needed for the logistic-map sequence

$ x_{k+1} = a x_k (1 - x_k) $

for various values of $a$.

For $1 < a < 3$, the sequence converges to a stable fixed point. As $a$ approaches 3 from below, convergence slows. We sample $a$ from 2.5 to 2.999 so successive outer-loop iterations have increasing workloads while remaining below the period-doubling transition.

We call the sequence converged when $|x_{k+1} - x_k| < \mathrm{tol}$. The program stops after `max_iter` iterations if this condition is not met.

This recurrence is the [logistic map](https://en.wikipedia.org/wiki/Logistic_map); its behavior depends on $a$.


In [ ]:
%%writefile omp_sched_rec.cc
#include <err.h>
#include <math.h>
#include <sched.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>
#include <omp.h>

/* return current time in nanoseconds */
long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}

/* record structure for each parameter value */
typedef struct {
  long c;
  long t[2];
  int thread[2];
  int cpu[2];
} record_t;

/*
   Iterate x <- a*x*(1-x), starting from x0, until it converges
   sufficiently close to the stable fixed point x* = 1 - 1/a.
   Return the number of iterations, or max_iter if not converged.
*/
long steps_to_converge(double a, double tolerance, long max_iter) {
  double x = 0.1;
  for (long i = 1; i <= max_iter; i++) {
    double x_ = a * x * (1.0 - x);
    if (fabs(x - x_) < tolerance) {
      return i;
    }
    x = x_;
  }
  return max_iter;
}

void record_logistic_map(double a, double tolerance, long max_iter, record_t *R) {
  R->thread[0] = omp_get_thread_num();
  R->cpu[0] = sched_getcpu();
  R->t[0] = cur_time_ns();
  R->c = steps_to_converge(a, tolerance, max_iter);
  R->t[1] = cur_time_ns();
  R->thread[1] = omp_get_thread_num();
  R->cpu[1] = sched_getcpu();
}

void dump(record_t *R, double a_min, double a_max, long n, long t0, long t1) {
  printf("T0=%ld T1=%ld\n", t0, t1);
  for (long i = 0; i < n; i++) {
    double a = a_min + (a_max - a_min) * (i + 0.5) / n;
    printf("a=%f c=%ld thread0=%d cpu0=%d thread1=%d cpu1=%d %ld %ld\n",
           a, R[i].c, R[i].thread[0], R[i].cpu[0],
           R[i].thread[1], R[i].cpu[1], R[i].t[0] - t0, R[i].t[1] - t0);
  }
}

int main(int argc, char **argv) {
  int idx = 1;
  /* number of points to sample in the interval [a_min, a_max] */
  long n = (idx < argc ? atol(argv[idx]) : 1000); idx++;
  /* minimum/maximum value of a */
  double a_min = (idx < argc ? atof(argv[idx]) : 2.5); idx++;
  double a_max = (idx < argc ? atof(argv[idx]) : 2.999); idx++;
  /* |x_n+1 - x_n| < tolerance => converged */
  double tolerance = (idx < argc ? atof(argv[idx]) : 1.0e-11); idx++;
  /* quit after this number of iterations when not converged */
  long max_iter = (idx < argc ? atol(argv[idx]) : 10000); idx++;
  if (a_min < 0 || a_max <= a_min || max_iter <= 0) {
    errx(1, "expected 0 <= a_min < a_max and max_iter > 0");
  }

  record_t *R = (record_t *)calloc(n, sizeof(record_t));
  if (!R) err(1, "calloc");

  long t0 = cur_time_ns();
#pragma omp parallel for schedule(runtime)
  for (long i = 0; i < n; i++) {
    double a = a_min + (a_max - a_min) * (i + 0.5) / n;
    record_logistic_map(a, tolerance, max_iter, &R[i]);
  }
  long t1 = cur_time_ns();

  dump(R, a_min, a_max, n, t0, t1);
  free(R);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp -D_GNU_SOURCE omp_sched_rec.cc -o omp_sched_rec
# nvc++ -O4 -Wall -mp -D_GNU_SOURCE omp_sched_rec.cc -o omp_sched_rec

In [ ]:
%%sub
# ./omp_sched_rec n a_min a_max tol max_iter
OMP_NUM_THREADS=4 OMP_SCHEDULE=static ./omp_sched_rec 1000 2.5 2.999 1e-8 100000 > a.dat

## 9-2. Visualizing iteration -> thread mapping
* <font color="green">Exercise: Run the plotting cell to visualize the schedule.</font>
* The graph's horizontal axis is elapsed time in nanoseconds, its vertical axis is the parameter value $a$, and its colors identify the executing thread.


In [ ]:
#!/usr/bin/python3
import re

import matplotlib.collections as mc
import matplotlib.pyplot as plt
import numpy as np

def read_dat(files_dat):
    # a=4.975000 c=100000 thread0=0 cpu0=6 thread1=0 cpu1=6 20448858 21018118
    pat0 = re.compile(r"T0=(?P<t0>\d+) T1=(?P<t1>\d+)")
    pat = re.compile(r"a=(?P<a>\d+\.\d+) c=(?P<c>\d+) thread0=(?P<thread0>\d+) cpu0=(?P<cpu0>\d+) thread1=(?P<thread1>\d+) cpu1=(?P<cpu1>\d+)(?P<t>( \d+)*)")
    log = {}
    t0 = None
    t1 = None
    for file_dat in files_dat:
        with open(file_dat) as fp:
            for i, line in enumerate(fp):
                if t0 is None:
                    m = pat0.match(line)
                    t0 = int(m.group("t0"))
                    t1 = int(m.group("t1"))
                    continue
                # 1 : 100.000000 20524414966449 20524423007875 0 0
                m = pat.match(line)
                if not m:
                    continue
                a = float(m.group("a"))
                thread0 = int(m.group("thread0"))
                thread1 = int(m.group("thread1"))
                cpu0 = int(m.group("cpu0"))
                cpu1 = int(m.group("cpu1"))
                c      = int(m.group("c"))
                t      = [int(s) for s in m.group("t").strip().split()]
                assert(thread0 == thread1), (thread0, thread1)
                if thread0 not in log:
                    log[thread0] = []
                log[thread0].append((a, t))
    return log, (t1 - t0)

def sched_plt(files_dat, start_t=0, end_t=float("inf"), start_a=0, end_a=float("inf")):
    log, t1 = read_dat(files_dat)
    n_threads = max(thread for thread in log) + 1
    # cmap = plt.cm.get_cmap('RdYlGn', n_threads)
    cmap = plt.get_cmap('RdYlGn', n_threads)
    fig, ax = plt.subplots()
    plt.xlabel("time (ns)")
    plt.ylabel("a")
    T0 = min(t for records in log.values() for _, T in records for t in T[:1])
    A0 = min(a for records in log.values() for a, _ in records)
    A1 = max(a for records in log.values() for a, _ in records)
    for thread,records in sorted(list(log.items())):
        # we take clocks relatie to the start (minimum) clock in each thread
        # T0 = min(t for _, T in records for t in T[:1])
        X = []
        Y = []
        thread_color = cmap(thread)
        for a, T in records:
            if start_a <= a < end_a:
                for t in T:
                    if start_t <= t - T0 <= end_t:
                        X.append(t - T0)
                        Y.append(a)
        ax.plot(X, Y, 'o', markersize=0.5, color=thread_color)
    ax.plot([t1, t1], [A0, A1])
    ax.autoscale()
    plt.savefig("sched.svg")
    plt.show()

In [ ]:
sched_plt(["a.dat"])

## 9-3. Specifying the scheduling policy by schedule clause
* <font color="green">Exercise: Compare `static` and `dynamic` scheduling. Explain how the schedules change the distribution of work and execution time.  Say which one is better in this case and explain why.</font>
* The loop uses `schedule(runtime)`, so you can select a policy at run time with `OMP_SCHEDULE`.


In [ ]:
%%sub
OMP_NUM_THREADS=4 OMP_SCHEDULE=dynamic ./omp_sched_rec 1000 2.5 2.999 1e-8 100000 > b.dat

In [ ]:
sched_plt(["b.dat"])

# <font color="green"> Problem 1 :  The reasons for canonical form restriction </font>
* What do you think is the reason(s) for restricting the `for` loop to the canonical form?
* This is basically for simplicity of implementing scheduling
* Elaborate this



<font color="green">WRITE YOUR ANSWER HERE (double click on the cell to edit)</font>_

...


# 10. Task parallelism
* A task is a unit of work that the OpenMP runtime schedules dynamically on a thread in a team created by `#pragma omp parallel`.
* A thread can create a task at any point in a parallel region. The runtime schedules created tasks on threads in the team.
* A task can create more tasks, so task parallelism can be nested and the total number of tasks may be difficult to predict (unlike a loop's iteration count).
* A common pattern is:
  1. Enter a parallel region with `#pragma omp parallel`.
  1. Use [#pragma omp masked](https://www.openmp.org/spec-html/5.2/openmpse61.html) that executes the statement below it by thread 0.  Use `filter` clause to select other thread(s) explicitly.
  1. Create tasks with [#pragma omp task](https://www.openmp.org/spec-html/5.2/openmpse73.html).
  1. Use [#pragma omp taskwait](https://www.openmp.org/spec-html/5.2/openmpse93.html) to wait for child tasks.

* First, observe the effect of `#pragma omp masked` without creating tasks.


In [ ]:
%%writefile omp_masked.cc
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel
  {
#pragma omp masked
    printf("inside the masked pragma: I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
    printf("out of the masked pragma: I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_masked.cc -o omp_masked
# nvc++ -O4 -Wall -mp omp_masked.cc -o omp_masked

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_masked

* `masked` pragma can be combined with `parallel` as `#pragma omp parallel masked`. 
  * <font color="red">This combined construct is not supported by the NVIDIA compiler used in this tutorial.</font>


In [ ]:
%%writefile omp_parallel_masked.cc
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel masked
  printf("I am thread %d of a team of %d threads\n",
         omp_get_thread_num(), omp_get_num_threads());
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_parallel_masked.cc -o omp_parallel_masked
# nvc++ -O4 -Wall -mp omp_parallel_masked.cc -o omp_parallel_masked

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_parallel_masked

* Now create a few tasks in `parallel` region.


In [ ]:
%%writefile omp_task.cc
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel
#pragma omp masked
  {
    printf("I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
#pragma omp task
    {
      printf("task A executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
      usleep(500 * 1000);
    }
#pragma omp task
    {
      printf("task B executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
      usleep(1000 * 1000);
    }
#pragma omp taskwait
    printf("two tasks done, executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_task.cc -o omp_task
# nvc++ -O4 -Wall -mp omp_task.cc -o omp_task

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_task

* Tasks are useful for parallel recursion, as the following program demonstrates.
* This pattern appears in many algorithms, particularly divide-and-conquer algorithms.
* A loop can be expressed with tasks by recursively splitting its iteration interval.
* The following is a task-parallel version of the logistic-map workload used above.


In [ ]:
%%writefile omp_task_rec.cc
#include <err.h>
#include <math.h>
#include <sched.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>
#include <omp.h>

/* return current time in nanoseconds */
long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}

/* record structure for each parameter value */
typedef struct {
  long c;
  long t[2];
  int thread[2];
  int cpu[2];
} record_t;

/*
   Iterate x <- a*x*(1-x), starting from x0, until it converges
   sufficiently close to the stable fixed point x* = 1 - 1/a.
   Return the number of iterations, or max_iter if not converged.
*/
long steps_to_converge(double a, double tolerance, long max_iter) {
  double x = 0.1;
  for (long i = 1; i <= max_iter; i++) {
    double x_ = a * x * (1.0 - x);
    if (fabs(x - x_) < tolerance) {
      return i;
    }
    x = x_;
  }
  return max_iter;
}

void record_logistic_map(double a, double tolerance, long max_iter, record_t *R) {
  R->thread[0] = omp_get_thread_num();
  R->cpu[0] = sched_getcpu();
  R->t[0] = cur_time_ns();
  R->c = steps_to_converge(a, tolerance, max_iter);
  R->t[1] = cur_time_ns();
  R->thread[1] = omp_get_thread_num();
  R->cpu[1] = sched_getcpu();
}

void record_logistic_map_rec(double a_min, double a_max, long i, long j, long n,
			     double tolerance, long max_iter, record_t * R) {
  if (j - i == 1) {
    double a = a_min + (a_max - a_min) * (i + 0.5) / n;
    record_logistic_map(a, tolerance, max_iter, &R[i]);
  } else {
    long k = (i + j) / 2;
#pragma omp task
    record_logistic_map_rec(a_min, a_max, i, k, n, tolerance, max_iter, R);
#pragma omp task
    record_logistic_map_rec(a_min, a_max, k, j, n, tolerance, max_iter, R);
#pragma omp taskwait
  }
}

void dump(record_t *R, double a_min, double a_max, long n, long t0, long t1) {
  printf("T0=%ld T1=%ld\n", t0, t1);
  for (long i = 0; i < n; i++) {
    double a = a_min + (a_max - a_min) * (i + 0.5) / n;
    printf("a=%f c=%ld thread0=%d cpu0=%d thread1=%d cpu1=%d %ld %ld\n",
           a, R[i].c, R[i].thread[0], R[i].cpu[0],
           R[i].thread[1], R[i].cpu[1], R[i].t[0] - t0, R[i].t[1] - t0);
  }
}

int main(int argc, char **argv) {
  int idx = 1;
  /* number of points to sample in the interval [a_min, a_max] */
  long n = (idx < argc ? atol(argv[idx]) : 1000); idx++;
  /* minimum/maximum value of a */
  double a_min = (idx < argc ? atof(argv[idx]) : 2.5); idx++;
  double a_max = (idx < argc ? atof(argv[idx]) : 2.999); idx++;
  /* |x_n+1 - x_n| < tolerance => converged */
  double tolerance = (idx < argc ? atof(argv[idx]) : 1.0e-11); idx++;
  /* quit after this number of iterations when not converged */
  long max_iter = (idx < argc ? atol(argv[idx]) : 10000); idx++;
  if (a_min < 0 || a_max <= a_min || max_iter <= 0) {
    errx(1, "expected 0 <= a_min < a_max and max_iter > 0");
  }

  record_t *R = (record_t *)calloc(n, sizeof(record_t));
  if (!R) err(1, "calloc");

  long t0 = cur_time_ns();
#pragma omp parallel masked
  record_logistic_map_rec(a_min, a_max, 0, n, n, tolerance, max_iter, R);
  long t1 = cur_time_ns();

  dump(R, a_min, a_max, n, t0, t1);
  free(R);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_task_rec.cc -o omp_task_rec
# nvc++ -O4 -Wall -mp omp_task_rec.cc -o omp_task_rec

In [ ]:
%%sub
# ./omp_task_rec n a_min a_max tol max_iter
OMP_NUM_THREADS=4 ./omp_task_rec 1000 2.5 2.999 1e-8 100000 > a.dat

In [ ]:
sched_plt(["a.dat"])
# sched_plt(["a.dat"], start_t=4e6, end_t=6e6)

# <font color="green"> Problem 2 :  What are recursive tasks useful for?</font>
* <font color="green">Name a few algorithms that could benefit from recursive tasks and explain how they work.</font>



<font color="green">WRITE YOUR ANSWER HERE (double click on the cell to edit)</font>_

* ...
* ...


# 11. Data sharing
## 11-1. Basics
* OpenMP uses a _shared-memory_ programming model. Variables declared outside a parallel region are generally shared by its threads unless a data-sharing rule or clause says otherwise.
* Threads share the same storage for shared variables. Synchronization is still required when one thread writes data that another thread reads; unsynchronized conflicting accesses can be a data race.
* This applies to global variables, local variables declared outside the region, and heap objects reached through shared pointers. Variables declared inside the region are private by default.
* See the specification's [data environment](https://www.openmp.org/spec-html/5.2/openmpch5.html) chapter.
* The following examples illustrate these rules:
  * `gv`, `ga`, `lv`, `la`, and the object pointed to by `ha` are shared in the base example.
  * `tid` is a separate automatic variable for each thread because it is declared inside the parallel region.

* <font color="green">Exercise: Run the local-variable example. Observe that the threads share `lv` and `la`, and that the main thread can observe the updates made in the parallel region.</font>


In [ ]:
%%writefile omp_data_basics_local.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>
int main() {
  int lv = 12;
  int la[N] = {34, 56, 78};
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : lv = %d, la = { %d, %d, %d }\n", id, lv, la[0], la[1], la[2]);
    if (id == 1) { lv += 100; }
    if (id < 3)  { la[id] += 100; }
  }
  printf("after : lv = %d, la = { %d, %d, %d }\n", lv, la[0], la[1], la[2]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_basics_local.cc -o omp_data_basics_local
# nvc++ -O4 -Wall -mp omp_data_basics_local.cc -o omp_data_basics_local

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_basics_local

* <font color="green">Exercise: Run the global-variable example and compare its behavior with the local-variable example.</font>


In [ ]:
%%writefile omp_data_basics_global.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int gv = 12;
int ga[N] = {34, 56, 78};
int main() {
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : gv = %d, ga = { %d, %d, %d }\n", id, gv, ga[0], ga[1], ga[2]);
    if (id == 1) { gv += 100; }
    if (id < 3)  { ga[id] += 100; }
  }
  printf("after : gv = %d, ga = { %d, %d, %d }\n", gv, ga[0], ga[1], ga[2]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_basics_global.cc -o omp_data_basics_global
# nvc++ -O4 -Wall -mp omp_data_basics_global.cc -o omp_data_basics_global

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_basics_global

* <font color="green">Exercise: Run the heap-allocation example. Observe that sharing a pointer also shares access to the object it points to.</font>


In [ ]:
%%writefile omp_data_basics_heap.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>
int main() {
  int * ha = new int[N] {34, 56, 78};
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : ha = { %d, %d, %d }\n", id, ha[0], ha[1], ha[2]);
    if (id < 3)  { ha[id] += 100; }
  }
  printf("after : ha = { %d, %d, %d }\n", ha[0], ha[1], ha[2]);
  delete[] ha;
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_basics_heap.cc -o omp_data_basics_heap
# nvc++ -O4 -Wall -mp omp_data_basics_heap.cc -o omp_data_basics_heap

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_basics_heap

* Sometimes each thread needs its own copy of a variable.
* The `private` clause on `#pragma omp parallel` creates a separate private copy for each thread.

## 11-2. `private` clause
* `private(x)` gives each thread its own `x`, as if it were declared inside the parallel region.
* A private copy is not initialized from the original value. Initialize it inside the region before reading it; reading an uninitialized copy has undefined behavior.
* <font color="green">Exercise: Add the appropriate `private` clause to each example, observe what happens, and reason about it according to the spec.</font>

* Local variables and arrays:


In [ ]:
%%writefile omp_data_private_local.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>
int main() {
  int lv = 12;
  int la[N] = {34, 56, 78};
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : lv = %d, la = { %d, %d, %d }\n", id, lv, la[0], la[1], la[2]);
    if (id == 1) { lv += 100; }
    if (id < 3)  { la[id] += 100; }
  }
  printf("after : lv = %d, la = { %d, %d, %d }\n", lv, la[0], la[1], la[2]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_private_local.cc -o omp_data_private_local
# nvc++ -O4 -Wall -mp omp_data_private_local.cc -o omp_data_private_local

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_private_local

* Global variables and arrays:


In [ ]:
%%writefile omp_data_private_global.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int gv = 12;
int ga[N] = {34, 56, 78};
int main() {
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : gv = %d, ga = { %d, %d, %d }\n", id, gv, ga[0], ga[1], ga[2]);
    if (id == 1) { gv += 100; }
    if (id < 3)  { ga[id] += 100; }
  }
  printf("after : gv = %d, ga = { %d, %d, %d }\n", gv, ga[0], ga[1], ga[2]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_private_global.cc -o omp_data_private_global
# nvc++ -O4 -Wall -mp omp_data_private_global.cc -o omp_data_private_global

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_private_global

* Heap-allocated variables and arrays:


In [ ]:
%%writefile omp_data_private_heap.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>
int main() {
  int * ha = new int[N] {34, 56, 78};
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : ha = { %d, %d, %d }\n", id, ha[0], ha[1], ha[2]);
    if (id < 3)  { ha[id] += 100; }
  }
  printf("after : ha = { %d, %d, %d }\n", ha[0], ha[1], ha[2]);
  delete[] ha;
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_private_heap.cc -o omp_data_private_heap
# nvc++ -O4 -Wall -mp omp_data_private_heap.cc -o omp_data_private_heap

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_private_heap

## 11-3. `firstprivate` clause
* `firstprivate(x)` is like `private(x)`, except each thread's private copy is initialized from the value immediately before the parallel region.
* For a pointer, `firstprivate` copies the pointer value, not the pointed-to object.
* <font color="green">Exercise: Add the appropriate `firstprivate` clause to each example, observe what happens, and reason about it according to the spec.</font>

* Local variables and arrays


In [ ]:
%%writefile omp_data_firstprivate_local.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>
int main() {
  int lv = 12;
  int la[N] = {34, 56, 78};
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : lv = %d, la = { %d, %d, %d }\n", id, lv, la[0], la[1], la[2]);
    if (id == 1) { lv += 100; }
    if (id < 3)  { la[id] += 100; }
  }
  printf("after : lv = %d, la = { %d, %d, %d }\n", lv, la[0], la[1], la[2]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_firstprivate_local.cc -o omp_data_firstprivate_local
# nvc++ -O4 -Wall -mp omp_data_firstprivate_local.cc -o omp_data_firstprivate_local

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_firstprivate_local

* Global variables and arrays


In [ ]:
%%writefile omp_data_firstprivate_global.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int gv = 12;
int ga[N] = {34, 56, 78};
int main() {
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : gv = %d, ga = { %d, %d, %d }\n", id, gv, ga[0], ga[1], ga[2]);
    if (id == 1) { gv += 100; }
    if (id < 3)  { ga[id] += 100; }
  }
  printf("after : gv = %d, ga = { %d, %d, %d }\n", gv, ga[0], ga[1], ga[2]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_firstprivate_global.cc -o omp_data_firstprivate_global
# nvc++ -O4 -Wall -mp omp_data_firstprivate_global.cc -o omp_data_firstprivate_global

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_firstprivate_global

* Heap-allocated variables and arrays


In [ ]:
%%writefile omp_data_firstprivate_heap.cc
enum { N = 3 };
#include <stdio.h>
#include <unistd.h>
#include <omp.h>
int main() {
  int * ha = new int[N] {34, 56, 78};
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : ha = { %d, %d, %d }\n", id, ha[0], ha[1], ha[2]);
    if (id < 3)  { ha[id] += 100; }
  }
  printf("after : ha = { %d, %d, %d }\n", ha[0], ha[1], ha[2]);
  delete[] ha;
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_data_firstprivate_heap.cc -o omp_data_firstprivate_heap
# nvc++ -O4 -Wall -mp omp_data_firstprivate_heap.cc -o omp_data_firstprivate_heap

In [ ]:
%%sub
OMP_NUM_THREADS=3 ./omp_data_firstprivate_heap

# 12. Data race
* <font color="green">Exercise: Run the following program and check if it correctly count the primes in $[2,n)$</font>


In [ ]:
%%writefile omp_race.cc
#include <err.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>
#include <omp.h>

long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}

long is_prime(long n) {
  for (long i = 2; i * i <= n; i++) {
    if (n % i == 0) return 0;
  }
  return 1;
}

int main(int argc, char **argv) {
  long n = (argc > 1) ? atol(argv[1]) : 1000000;
  long s = 0;
  long t0 = cur_time_ns();
#pragma omp parallel for
  for (long i = 2; i < n; i++) {
    if (is_prime(i)) {
      s++;
    }
  }
  long t1 = cur_time_ns();
  printf("Number of primes in [2,%ld): %ld\n", n, s);
  printf("Time taken: %f s\n", (double)(t1 - t0) * 1e-9);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_race.cc -o omp_race
# nvc++ -O4 -Wall -mp omp_race.cc -o omp_race

In [ ]:
%%sub
OMP_NUM_THREADS=18 ./omp_race 1000000

* With one thread, or without `#pragma omp parallel for`, the program gives the correct result.
* With multiple threads, the result may be wrong and may vary between runs because the updates to `s` race.

# <font color="green"> Problem 3 :  What can go wrong with this program?</font>
* Explain why the result can be incorrect or vary between runs



<font color="green">WRITE YOUR ANSWER HERE (double click on the cell to edit)</font>_

...


* More generally, a _data race_ occurs when multiple threads access the same memory location concurrently, at least one access is a write, and no synchronization orders the accesses. In C++, a data race causes undefined behavior.

* In most parallel programs, threads need to communicate to accomplish a task.
* Threads can communicate when one thread writes a variable and another reads it.
* Without appropriate synchronization, conflicting accesses can create a data race.

* The following constructs can coordinate shared updates safely:

  * `#pragma omp critical`
  * `#pragma omp atomic`
  * reduction

## 12-1. `#pragma omp critical`
* [#pragma omp critical](https://www.openmp.org/spec-html/5.2/openmpse90.html) ensures that only one thread at a time executes the associated structured block for a given critical-region name.

* <font color="green">Exercise: Add `#pragma omp critical` around `s++` and check the result.</font>


In [ ]:
%%writefile omp_critical.cc
#include <err.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>
#include <omp.h>

long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}

long is_prime(long n) {
  for (long i = 2; i * i <= n; i++) {
    if (n % i == 0) return 0;
  }
  return 1;
}

int main(int argc, char **argv) {
  long n = (argc > 1) ? atol(argv[1]) : 1000000;
  long s = 0;
  long t0 = cur_time_ns();
#pragma omp parallel for
  for (long i = 2; i < n; i++) {
    if (is_prime(i)) {
      s++;
    }
  }
  long t1 = cur_time_ns();
  printf("Number of primes in [2,%ld): %ld\n", n, s);
  printf("Time taken: %f s\n", (double)(t1 - t0) * 1e-9);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_critical.cc -o omp_critical
# nvc++ -O4 -Wall -mp omp_critical.cc -o omp_critical

In [ ]:
%%sub
OMP_NUM_THREADS=18 ./omp_critical 1000000

## 12-2. `#pragma omp atomic`
* [#pragma omp atomic](https://www.openmp.org/spec-html/5.2/openmpsu125.html) protects a supported memory operation. Its effect differs from `critical`, and it applies only to specific statement forms.

* <font color="green">Exercise: Use `#pragma omp atomic` on `s++`, then compare the result and runtime.</font>


In [ ]:
%%writefile omp_atomic.cc
#include <err.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>
#include <omp.h>

long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}

long is_prime(long n) {
  for (long i = 2; i * i <= n; i++) {
    if (n % i == 0) return 0;
  }
  return 1;
}

int main(int argc, char **argv) {
  long n = (argc > 1) ? atol(argv[1]) : 1000000;
  long s = 0;
  long t0 = cur_time_ns();
#pragma omp parallel for
  for (long i = 2; i < n; i++) {
    if (is_prime(i)) {
      s++;
    }
  }
  long t1 = cur_time_ns();
  printf("Number of primes in [2,%ld): %ld\n", n, s);
  printf("Time taken: %f s\n", (double)(t1 - t0) * 1e-9);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_atomic.cc -o omp_atomic
# nvc++ -O4 -Wall -mp omp_atomic.cc -o omp_atomic

In [ ]:
%%sub
OMP_NUM_THREADS=18 ./omp_atomic 1000000

* The statement associated with this directive must have one of the forms permitted by the specification; it cannot be an arbitrary expression.
* See the specification's [atomic construct](https://www.openmp.org/spec-html/5.2/openmpsu125.html).
* A common use is to update a variable, for example:
```  
x += expr;
```  
* The guarantees of `#pragma omp atomic` differ from those of `#pragma omp critical`:
```
#pragma omp atomic
x += expr;
``` 
ensures that the specified atomic update to `x` is indivisible with respect to other atomic accesses to the same location, whereas
```
#pragma omp critical
x += expr;
``` 
ensures that execution of the entire critical region does not overlap with another thread executing the same named critical region.
* When applicable, `#pragma omp atomic` can be more efficient than `critical` because it protects a narrower operation.

## 12-3. `reduction` clause
* A [reduction](https://www.openmp.org/spec-html/5.2/openmpsu52.html) is often the preferred way to combine independent partial results.
* It applies when threads compute $v = v_0 \oplus v_1 \oplus \dots \oplus v_{n-1}$, where each $v_i$ is computed independently and $\oplus$ is an associative operator, such as integer addition.
* A serial loop might be written as:
```
v = initial value;
for (i = 0; i < n; i++) {
  v_i = ...
  v = v (+) v_i;
}
```
* Parallelizing the loop without synchronization would make updates to $v$ race.
* Add `reduction(+ : v)` to the parallel construct to combine thread-local partial sums safely.

* <font color="green">Exercise: Add a `reduction` clause to the parallel construct and check the result.</font>


In [ ]:
%%writefile omp_reduction.cc
#include <err.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>
#include <omp.h>

long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}

long is_prime(long n) {
  for (long i = 2; i * i <= n; i++) {
    if (n % i == 0) return 0;
  }
  return 1;
}

int main(int argc, char **argv) {
  long n = (argc > 1) ? atol(argv[1]) : 1000000;
  long s = 0;
  long t0 = cur_time_ns();
#pragma omp parallel for
  for (long i = 2; i < n; i++) {
    if (is_prime(i)) {
      s++;
    }
  }
  long t1 = cur_time_ns();
  printf("Number of primes in [2,%ld): %ld\n", n, s);
  printf("Time taken: %f s\n", (double)(t1 - t0) * 1e-9);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_reduction.cc -o omp_reduction
# nvc++ -O4 -Wall -mp omp_reduction.cc -o omp_reduction

In [ ]:
%%sub
OMP_NUM_THREADS=18 ./omp_reduction

## 12-4. How the reduction clause works and why it is often preferable
* Where applicable, a reduction is generally faster than repeatedly using `#pragma omp atomic` or `#pragma omp critical`.
* Each thread computes a partial result in a private variable; the partial results are combined at the end.
* For example, `x += expr` updates a thread's private copy of `x` rather than the shared variable.
* Informally, the reduction transforms code like:

```
int v = G;
#pragma omp parallel reduction(+ : v)
{
   ... v += expr; ...
}
  
```

into something like

```
int v = G;
#pragma omp parallel
{
  int v_priv = 0; // (I) initialize each private partial result
  {
   ... v_priv += expr; ...
  }
#pragma omp atomic
  v += v_priv; // (C) combine this thread's partial result with the original value
}
```
* This illustration assumes an associative operation and omits implementation details.

## 12-5. User-defined reduction
* Reduction is a general method for combining independently computed values $v_i$ into $v = v_0 \oplus v_1 \oplus \dots \oplus v_{n-1}$.
* It is applicable when the order of combining partial results with $\oplus$ does not affect the result, as with associative operations such as integer addition.
* Built-in reductions support selected operations and types. User-defined reductions extend this mechanism to user-defined types.
* [User-defined reductions (`declare reduction`)](https://www.openmp.org/spec-html/5.2/openmpsu55.html) let you define reductions for your own types.
* Define two operations:
  * (C) combine the value in `omp_in` into `omp_out`.
  * (I) initialize the thread-private value `omp_priv` to the identity element.
* For addition, these are `omp_out += omp_in` and `omp_priv = 0`.

## 12-6. Apply a user-defined reduction
* The following program attempts to reduce a three-element vector, but its parallel loop is incorrect.
* <font color="green">Exercise: Define a reduction with `#pragma omp declare reduction` and apply it to the loop.</font>


In [ ]:
%%writefile omp_ud_reduction.cc
#include <stdio.h>
#include <unistd.h>
#include <math.h>
#include <omp.h>

/* 3-element vector */
typedef struct {
  double a[3];
} vec_t;

/* x += y */
void vec_add(vec_t * x, vec_t * y) {
  for (int i = 0; i < 3; i++) {
    x->a[i] += y->a[i];
  }
}

/* x = {0,0,0} */
void vec_init(vec_t * x) {
  for (int i = 0; i < 3; i++) {
    x->a[i] = 0;
  }
}


/* add an appropriate #pragma omp declare reduction ... here */
  
int main() {
  vec_t v;
  vec_init(&v);
  double t0 = omp_get_wtime();
  /* add an appropriate reduction clause, so that
     the result is always {10000,10000,10000} */
#pragma omp parallel for
  for (int i = 0; i < 30000; i++) {
    v.a[i % 3]++;
  }
  double t1 = omp_get_wtime();
  printf("ans = {%.1f, %.1f, %.1f} in %f sec\n", v.a[0], v.a[1], v.a[2], t1 - t0);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O3 -Wall -fopenmp omp_ud_reduction.cc -o omp_ud_reduction
# nvc++ -O4 -Wall -mp omp_ud_reduction.cc -o omp_ud_reduction

In [ ]:
%%sub
OMP_NUM_THREADS=18 ./omp_ud_reduction